In [ ]:
%pip install torch_geometric
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
import torch_geometric.transforms as T

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    f1_score, precision_recall_curve
)
from xgboost import XGBClassifier
# --- INSERT THIS AFTER IMPORTS ---
def build_user_features(df, user_col="User"):
    agg = df.groupby(user_col).agg(
        txn_count = ("Amount", "count"),
        mean_amount = ("Amount", "mean"),
        std_amount = ("Amount", "std"),
        # REMOVED fraud_rate to prevent target leakage
        night_txn_ratio = ("Hour", lambda h: ((h < 6) | (h > 22)).mean()),
        online_ratio = ("chip_Online Transaction", "mean"), # Using your OHE columns
        swipe_ratio = ("chip_Swipe Transaction", "mean"),
        mcc_diversity = ("MCC", lambda m: m.nunique()),
        error_rate = ("err_No error", "mean") # Using your OHE columns
    ).reset_index()
    agg["txn_count"] = np.log1p(agg["txn_count"])
    return agg

def build_node_features(df, id_col):
    # Similar aggregation logic for cards and merchants
    agg = df.groupby(id_col).agg(
        txn_count = ("Amount", "count"),
        mean_amount = ("Amount", "mean"),
        std_amount = ("Amount", "std"),
    ).reset_index()
    agg["txn_count"] = np.log1p(agg["txn_count"])
    return agg

# Data

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "credit_card_transactions-ibm_v2.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "ealtman2019/credit-card-transactions",
  file_path,
)

Using Colab cache for faster access to the 'credit-card-transactions' dataset.


#### Downsampling

In [ ]:
non_fraud = df[df['Is Fraud?'] == 'No']
fraud = df[df['Is Fraud?'] == 'Yes']

non_fraud_reduced = non_fraud.sample(n=len(non_fraud)-23000000, random_state=42)

df = pd.concat([fraud, non_fraud_reduced]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['Is Fraud?'].value_counts())
print(df.head())

(1386900, 15)
Is Fraud?
No     1357143
Yes      29757
Name: count, dtype: int64
   User  Card  Year  Month  Day   Time  Amount            Use Chip  \
0  1560     1  2013      7   18  14:59  $20.24  Online Transaction   
1   488     0  2003      8   23  12:31  $15.43   Swipe Transaction   
2  1634     2  2019      8   30  19:09   $2.09    Chip Transaction   
3  1429     1  2018     11    2  14:29  $52.47  Online Transaction   
4  1276     0  2019      9    9  10:53  $37.68  Online Transaction   

         Merchant Name Merchant City Merchant State      Zip   MCC Errors?  \
0 -6458444334611773637        ONLINE            NaN      NaN  4784     NaN   
1  6621275923246397337   Los Angeles             CA  90033.0  5814     NaN   
2 -6571010470072147219      Meridian             MS  39301.0  5499     NaN   
3 -2088492411650162548        ONLINE            NaN      NaN  4784     NaN   
4  4241336128694185533        ONLINE            NaN      NaN  4814     NaN   

  Is Fraud?  
0        No  
1 

# Data cleaning

In [ ]:
# 1. Strip $ from Amount and convert to float
df['Amount'] = df['Amount'].str.replace('$', '', regex=False).astype(float)

# 2. Encode Is Fraud? to 0/1
df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})

# 3. Extract continuous decimal hour from Time
df['Hour'] = df['Time'].str.split(':').str[0].astype(int)
df['Minute'] = df['Time'].str.split(':').str[1].astype(int)
df['Decimal_Hour'] = df['Hour'] + (df['Minute'] / 60.0) # Capture minutes!
df.drop(columns=['Time', 'Minute'], inplace=True)

# One-hot encode Use Chip
use_chip_dummies = pd.get_dummies(df['Use Chip'], prefix='chip').astype(np.float32)
df = pd.concat([df, use_chip_dummies], axis=1)
df.drop(columns=['Use Chip'], inplace=True)   # drop original string column, same as Errors?

df["card_id"] = df["User"].astype(str) + "_" + df["Card"].astype(str)
df = df.drop(columns=['Merchant State', 'Card', 'Zip'])

# NEW: One-hot encode Errors? (Your EDA showed this is a massive predictor!)
df["Errors?"] = df["Errors?"].fillna("No error")
error_dummies = pd.get_dummies(df['Errors?'], prefix='err').astype(np.float32)
df = pd.concat([df, error_dummies], axis=1)
df.drop(columns=['Errors?'], inplace=True) # Drop original string column

# Train/Test split

In [ ]:
# 1. GLOBAL ID MAPPING (Before Split)
# We map all customers and merchants to indices using the FULL dataframe.
# This ensures that "Customer A" has the same node index in both the train graph
# and the test graph, allowing us to transfer features later.
customer_ids = df["card_id"].unique()
merchant_ids = df["Merchant Name"].unique()
user_ids = df["User"].unique()

customer_id_map = {cid: i for i, cid in enumerate(customer_ids)}
merchant_id_map = {mid: i for i, mid in enumerate(merchant_ids)}
user_id_map = {uid: i for i, uid in enumerate(user_ids)}

df["cust_node_idx"] = df["card_id"].map(customer_id_map)
df["merch_node_idx"] = df["Merchant Name"].map(merchant_id_map)
df["user_node_idx"] = df["User"].map(user_id_map)

# GLOBAL user -> card edge index (structural/ownership, not per-transaction —
# a card always belongs to the same user, so this is identical for train and test graphs)
card_owner_user_idx = (
    df.drop_duplicates("card_id")
      .set_index("card_id")["User"]
      .map(user_id_map)
      .reindex(customer_ids)
      .values
)
user_to_card_edge = torch.tensor(
    np.vstack([card_owner_user_idx, np.arange(len(customer_ids))]),
    dtype=torch.long
)
# 2. TEMPORAL SPLIT (Moved to the very top of graph building)
split_year = df['Year'].quantile(0.85)
print(f"Splitting data at year: {int(split_year)}")

train_df = df[df['Year'] < split_year].reset_index(drop=True)
test_df = df[df['Year'] >= split_year].reset_index(drop=True)
# 3. MCC FREQUENCY (fit on train only, apply to both — avoids leakage)
mcc_freq = train_df['MCC'].value_counts(normalize=True)
train_df['MCC_freq'] = train_df['MCC'].map(mcc_freq).astype(np.float32)
test_df['MCC_freq'] = test_df['MCC'].map(mcc_freq).fillna(0).astype(np.float32)

print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

Splitting data at year: 2018
Training samples: 1171626
Testing samples: 215274


## Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Sort data chronologically per user so shift() works properly
train_df = train_df.sort_values(by=['card_id', 'Year', 'Month', 'Day', 'Decimal_Hour']).reset_index(drop=True)
test_df = test_df.sort_values(by=['card_id', 'Year', 'Month', 'Day', 'Decimal_Hour']).reset_index(drop=True)

# 2. Add Velocity Features (Average amount of user's PREVIOUS 3 transactions)
# shift(1) ensures we only look at PAST transactions, preventing data leakage!
train_df['rolling_amount_3'] = train_df.groupby('card_id')['Amount'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
).fillna(0)
test_df['rolling_amount_3'] = test_df.groupby('card_id')['Amount'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
).fillna(0)

# 3. Scale all continuous columns
continuous_cols = ["Amount", "Hour", "Decimal_Hour", "rolling_amount_3"]

scaler = StandardScaler()
train_df[continuous_cols] = scaler.fit_transform(train_df[continuous_cols])
test_df[continuous_cols] = scaler.transform(test_df[continuous_cols])  # transform only, no fit

# --- INSERT THIS RIGHT BEFORE # Train Graph ---

# 1. Generate feature dataframes
user_feat_df = build_user_features(train_df, "User")
cust_feat_df = build_node_features(train_df, "card_id")
merch_feat_df = build_node_features(train_df, "Merchant Name")

# 2. ALIGN features to global IDs (Crucial for PyTorch Geometric!)
# We use the user_ids, customer_ids, and merchant_ids arrays you defined earlier
user_feat_df.set_index("User", inplace=True)
user_feat_aligned = user_feat_df.reindex(user_ids).fillna(0)

cust_feat_df.set_index("card_id", inplace=True)
cust_feat_aligned = cust_feat_df.reindex(customer_ids).fillna(0)

merch_feat_df.set_index("Merchant Name", inplace=True)
merch_feat_aligned = merch_feat_df.reindex(merchant_ids).fillna(0)

# 3. Scale and convert to Tensors
user_scaler = StandardScaler()
train_user_x = torch.tensor(
    user_scaler.fit_transform(user_feat_aligned.values),
    dtype=torch.float
)

cust_scaler = StandardScaler()
train_customer_x = torch.tensor(
    cust_scaler.fit_transform(cust_feat_aligned.values),
    dtype=torch.float
)

merch_scaler = StandardScaler()
train_merchant_x = torch.tensor(
    merch_scaler.fit_transform(merch_feat_aligned.values),
    dtype=torch.float
)

# Define train_transaction_x (Make sure feature_cols are defined)
feature_cols = [c for c in train_df.columns if c not in [
    'Is Fraud?', 'card_id', 'Merchant Name', 'User',
    'cust_node_idx', 'merch_node_idx', 'user_node_idx',
    'Merchant City', 'MCC' # Explicitly exclude string/categorical columns
]]
train_transaction_x = torch.tensor(train_df[feature_cols].values.astype(np.float32), dtype=torch.float)

In [ ]:
print(df.tail(10))

         User  Year  Month  Day  Amount        Merchant Name Merchant City  \
1386890  1154  2007      3   13  106.37 -3971935829054965805        Revere   
1386891   370  2007      5    2   32.36 -2088492411650162548        ONLINE   
1386892  1772  2014      1    7    8.77  7847619186796796084  Williamsburg   
1386893  1151  2007      5    1  -92.00  1799189980464955940      Hamilton   
1386894  1123  2017     12   18   18.38 -4500542936415012428   San Antonio   
1386895   896  2013      5    9    9.25  1874515938428798953    Louisville   
1386896  1488  2018      6    2  143.97 -5467922351692495955      Appleton   
1386897   697  2012      5   13   94.16  1913477460590765860        Vienna   
1386898   597  2011     11   25  -91.00  1799189980464955940       Atlanta   
1386899   777  2005      4    9   80.64 -8254405722253003769       Murdock   

          MCC  Is Fraud?  Hour  ...  err_Bad PIN,Insufficient Balance  \
1386890  5300          0     8  ...                               0.

In [ ]:
print(df.columns.tolist())

['User', 'Year', 'Month', 'Day', 'Amount', 'Merchant Name', 'Merchant City', 'MCC', 'Is Fraud?', 'Hour', 'Decimal_Hour', 'chip_Chip Transaction', 'chip_Online Transaction', 'chip_Swipe Transaction', 'card_id', 'err_Bad CVV', 'err_Bad CVV,Insufficient Balance', 'err_Bad CVV,Technical Glitch', 'err_Bad Card Number', 'err_Bad Card Number,Bad CVV', 'err_Bad Card Number,Bad Expiration', 'err_Bad Card Number,Insufficient Balance', 'err_Bad Expiration', 'err_Bad Expiration,Bad CVV', 'err_Bad Expiration,Insufficient Balance', 'err_Bad Expiration,Technical Glitch', 'err_Bad PIN', 'err_Bad PIN,Insufficient Balance', 'err_Bad PIN,Technical Glitch', 'err_Bad Zipcode', 'err_Insufficient Balance', 'err_Insufficient Balance,Technical Glitch', 'err_No error', 'err_Technical Glitch', 'cust_node_idx', 'merch_node_idx', 'user_node_idx']


In [ ]:
# Define train_transaction_x and feature_cols
feature_cols = [c for c in train_df.columns if c not in [
    'Is Fraud?', 'card_id', 'Merchant Name', 'User',
    'cust_node_idx', 'merch_node_idx', 'user_node_idx',
    'Merchant City', 'MCC' # Explicitly exclude string/categorical columns
]]
train_transaction_x = torch.tensor(train_df[feature_cols].values.astype(np.float32), dtype=torch.float)

# Train Graph

In [ ]:
import torch
import numpy as np
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

# 1. Combine dataframes to align global indices perfectly
combined_df = pd.concat([train_df, test_df]).reset_index(drop=True)

# 2. Global Transaction Features
global_transaction_x = torch.tensor(combined_df[feature_cols].values.astype(np.float32), dtype=torch.float)
global_y = torch.tensor(combined_df["Is Fraud?"].values, dtype=torch.float)

# 3. Global Edges
txn_idx_global = np.arange(len(combined_df))
cust_to_txn_global = torch.tensor(
    np.vstack([combined_df["cust_node_idx"].values, txn_idx_global]), dtype=torch.long
)
merch_to_txn_global = torch.tensor(
    np.vstack([combined_df["merch_node_idx"].values, txn_idx_global]), dtype=torch.long
)

# 4. Assemble Unified HeteroData
data = HeteroData()
data["customer"].x = train_customer_x
data["user"].x = train_user_x
data["merchant"].x = train_merchant_x
data["transaction"].x = global_transaction_x
data["transaction"].y = global_y

data["customer", "made", "transaction"].edge_index = cust_to_txn_global
data["merchant", "involved_in", "transaction"].edge_index = merch_to_txn_global
data["user", "has_card", "customer"].edge_index = user_to_card_edge

# This automatically creates all rev_* edges for you!
data = T.ToUndirected()(data)

# Test Graph

In [ ]:
# 5. Create Train/Test Masks (Crucial for preventing data leakage in the unified graph!)
train_mask = torch.tensor([True] * len(train_df) + [False] * len(test_df), dtype=torch.bool)
test_mask = torch.tensor([False] * len(train_df) + [True] * len(test_df), dtype=torch.bool)

data["transaction"].train_mask = train_mask
data["transaction"].test_mask = test_mask

print("Unified Graph with Masks:", data)

Unified Graph with Masks: HeteroData(
  customer={ x=[5971, 3] },
  user={ x=[1995, 8] },
  merchant={ x=[38892, 3] },
  transaction={
    x=[1386900, 30],
    y=[1386900],
    train_mask=[1386900],
    test_mask=[1386900],
  },
  (customer, made, transaction)={ edge_index=[2, 1386900] },
  (merchant, involved_in, transaction)={ edge_index=[2, 1386900] },
  (user, has_card, customer)={ edge_index=[2, 5971] },
  (transaction, rev_made, customer)={ edge_index=[2, 1386900] },
  (transaction, rev_involved_in, merchant)={ edge_index=[2, 1386900] },
  (customer, rev_has_card, user)={ edge_index=[2, 5971] }
)


# Model


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroSAGE(nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata, txn_in_dim, user_in_dim):
        super().__init__()
        edge_types = metadata[1]
        node_types = metadata[0]
        self.conv1 = HeteroConv(
            {et: SAGEConv((-1, -1), hidden_channels) for et in edge_types}, aggr="sum"
        )
        self.norm1 = nn.ModuleDict({nt: nn.LayerNorm(hidden_channels) for nt in node_types})
        self.conv2 = HeteroConv(
            {et: SAGEConv((-1, -1), out_channels) for et in edge_types}, aggr="sum"
        )
        self.norm2 = nn.ModuleDict({nt: nn.LayerNorm(out_channels) for nt in node_types})

        self.txn_skip = nn.Sequential(
            nn.Linear(txn_in_dim, out_channels),
            nn.LayerNorm(out_channels),
        )

        self.user_proj1 = nn.Linear(user_in_dim, hidden_channels)
        self.user_proj2 = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        txn_raw = x_dict["transaction"]
        user_raw = x_dict["user"]

        x_dict = self.conv1(x_dict, edge_index_dict)
        if "user" not in x_dict:
            x_dict["user"] = self.user_proj1(user_raw)

        x_dict = {k: F.relu(self.norm1[k](v)) for k, v in x_dict.items()}
        user_1 = x_dict["user"]

        x_dict = self.conv2(x_dict, edge_index_dict)
        if "user" not in x_dict:
            x_dict["user"] = self.user_proj2(user_1)

        x_dict = {k: self.norm2[k](v) for k, v in x_dict.items()}
        x_dict["transaction"] = x_dict["transaction"] + self.txn_skip(txn_raw)
        return x_dict

class FraudHead(nn.Module):
    def __init__(self, in_channels, hidden=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, transaction_emb):
        return self.net(transaction_emb).squeeze(-1)

# Masked VICReg to prevent target leakage during training
def vicreg_loss_multi(x_dict, mask, gamma=1.0, eps=1e-4):
    total = 0.0
    for k, z in x_dict.items():
        # ONLY apply to training transactions!
        if k == "transaction":
            z = z[mask]
        if z.size(0) < 2: continue

        std = z.std(dim=0) + eps
        cov = z - z.mean(dim=0, keepdim=True)
        var_loss = F.relu(1.0 - std).mean()
        cov_loss = (cov.t() @ cov / z.size(0)).fill_diagonal_(0).pow(2).mean()
        total = total + var_loss + cov_loss
    return gamma * total

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.9, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        pt = torch.exp(-bce_loss)
        alpha_t = torch.where(targets == 1, self.alpha, 1.2 - self.alpha)
        focal_loss = alpha_t * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

## Training

In [ ]:
# --- INITIALIZATION ---
encoder = HeteroSAGE(
    hidden_channels=128,
    out_channels=64,
    metadata=data.metadata(),
    txn_in_dim=data["transaction"].x.size(1),
    user_in_dim=data["user"].x.size(1)
)
head = FraudHead(64)

train_mask = data["transaction"].train_mask
y_train = data["transaction"].y[train_mask]

n_pos = (y_train == 1).sum().item()
n_neg = (y_train == 0).sum().item()
alpha = n_neg / (n_pos + n_neg)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=1e-3)
criterion = FocalLoss(alpha=alpha, gamma=2.0)

# --- TRAINING LOOP ---
for epoch in range(50):
    encoder.train(); head.train()
    optimizer.zero_grad()

    # Pass the ENTIRE unified graph through the network
    out_dict = encoder(data.x_dict, data.edge_index_dict)

    # APPLY MASK: Only calculate loss on the training nodes
    logits = head(out_dict["transaction"][train_mask])
    bce = criterion(logits, y_train)

    # LOWERED GAMMA: Drop vicreg weight to 0.005 to stop model collapse
    reg = vicreg_loss_multi(out_dict, mask=train_mask, gamma=0.005)
    loss = bce + reg

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(encoder.parameters()) + list(head.parameters()), max_norm=1.0
    )

    optimizer.step()

    if epoch % 5 == 0:
        with torch.no_grad():
            preds = (logits > 0).float()
            correct = (preds == y_train).sum().item()
            acc = correct / len(y_train)
            fraud_preds = preds.sum().item()
        print(f"Epoch {epoch:02d} | BCE: {bce.item():.4f} | Reg: {reg.item():.4f} | Train Acc: {acc:.4f} | Fraud Preds: {int(fraud_preds)}")

# Merge Embeddings

In [ ]:
from sklearn.preprocessing import StandardScaler

encoder.eval()
with torch.no_grad():
    # 1. Get embeddings for the ENTIRE graph at once
    out_dict = encoder(data.x_dict, data.edge_index_dict)
    all_embeddings = out_dict["transaction"].numpy()

    # 2. Slice back into train and test sets using our masks
    train_mask_np = data["transaction"].train_mask.numpy()
    test_mask_np = data["transaction"].test_mask.numpy()

    train_embeddings = all_embeddings[train_mask_np]
    test_embeddings = all_embeddings[test_mask_np]

# 3. Scale embeddings so XGBoost treats them fairly
emb_scaler = StandardScaler().fit(train_embeddings)

train_emb_df = pd.DataFrame(emb_scaler.transform(train_embeddings), columns=[f"gnn_emb_{i}" for i in range(train_embeddings.shape[1])])
test_emb_df = pd.DataFrame(emb_scaler.transform(test_embeddings), columns=[f"gnn_emb_{i}" for i in range(test_embeddings.shape[1])])

# 4. Merge with original tabular data
X_train = pd.concat([train_df.reset_index(drop=True), train_emb_df], axis=1)
X_test = pd.concat([test_df.reset_index(drop=True), test_emb_df], axis=1)

# Define targets
y_train = X_train['Is Fraud?']
y_test = X_test['Is Fraud?']
X_train = X_train.drop(columns=['Is Fraud?'])
X_test = X_test.drop(columns=['Is Fraud?'])

print("Train:", X_train.shape, "| Test:", X_test.shape)

# Train XGBoost

In [ ]:
scale_pos_weight = ((y_train == 0).sum() / (y_train == 1).sum()) * 2.0
print("scale_pos_weight:", scale_pos_weight)

# Drop non-numeric columns XGBoost can't handle, PLUS raw identity/index columns.
non_numeric_cols = X_train.select_dtypes(include=['object']).columns
id_leak_cols = ['card_id', 'Merchant Name', 'cust_node_idx', 'merch_node_idx', 'User', 'user_node_idx']
drop_cols = list(non_numeric_cols) + id_leak_cols
X_train_numeric = X_train.drop(columns=drop_cols, errors='ignore')
X_test_numeric = X_test.drop(columns=drop_cols, errors='ignore')

# NEW: Updated XGBoost parameters
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,            # shallower because embeddings are dense
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    early_stopping_rounds=30,  # stops training if it starts overfitting
    random_state=42,
    verbosity=0
)

model.fit(
    X_train_numeric,
    y_train,
    eval_set=[(X_test_numeric, y_test)],
    verbose=False
)

print("XGBoost training complete. Best iteration:", model.best_iteration)

# Evaluate

In [ ]:
y_pred = model.predict(X_test_numeric)
y_proba = model.predict_proba(X_test_numeric)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, digits=4))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

# Threshold Tuning

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

prec, rec, thresh = precision_recall_curve(y_test, y_proba)

# Original Tuning System: Find the threshold that gives the highest F1 score
f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1_curve.argmax()
best_thresh = thresh[best_idx]

print(f"Best threshold: {best_thresh:.4f} | Precision: {prec[best_idx]:.4f} | "
      f"Recall: {rec[best_idx]:.4f} | F1: {f1_curve[best_idx]:.4f}")

y_pred_tuned = (y_proba >= best_thresh).astype(int)
print("\n--- Classification Report (tuned threshold) ---")
print(classification_report(y_test, y_pred_tuned, digits=4))

# Ablation: GNN+XGBoost vs Raw XGBoost

In [ ]:
# Same leak-fixed feature set, same threshold-tuning procedure on both sides -- this is
# the direct, apples-to-apples test of whether the GNN embeddings are actually worth it,
# rather than relying on an earlier informal comparison at a fixed 0.5 threshold.
gnn_cols = [c for c in X_train_numeric.columns if c.startswith("gnn_emb_")]
X_train_raw = X_train_numeric.drop(columns=gnn_cols)
X_test_raw = X_test_numeric.drop(columns=gnn_cols)

model_raw = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    early_stopping_rounds=30,
    verbosity=0,
    random_state=42
)
model_raw.fit(X_train_raw, y_train, eval_set=[(X_test_raw, y_test)], verbose=False)

y_proba_raw = model_raw.predict_proba(X_test_raw)[:, 1]
prec_r, rec_r, thresh_r = precision_recall_curve(y_test, y_proba_raw)
f1_r = 2 * prec_r * rec_r / (prec_r + rec_r + 1e-9)
best_idx_r = f1_r.argmax()

print(f"Raw XGBoost  | PR-AUC: {average_precision_score(y_test, y_proba_raw):.4f} | "
      f"best-thresh precision: {prec_r[best_idx_r]:.4f} | recall: {rec_r[best_idx_r]:.4f}")
print(f"GNN+XGBoost  | PR-AUC: {average_precision_score(y_test, y_proba):.4f} | "
      f"best-thresh precision: {prec[best_idx]:.4f} | recall: {rec[best_idx]:.4f}")

In [ ]:
# 1. Check actual training class ratio
n_pos_train = (train_data["transaction"].y == 1).sum().item()
n_neg_train = (train_data["transaction"].y == 0).sum().item()
actual_ratio = n_neg_train / n_pos_train

#print(f"Hardcoded pos_weight: {pos_weight.item():.2f}")
print(f"Actual training ratio: {actual_ratio:.2f}")
#print(f"Mismatch: {actual_ratio / pos_weight.item():.2f}x wrong")

# 2. Visualize embedding separation
import numpy as np
from sklearn.manifold import TSNE

# Get embeddings
encoder.eval()
with torch.no_grad():
    train_out = encoder(train_data.x_dict, train_data.edge_index_dict)
    fraud_emb = train_out["transaction"][train_data["transaction"].y == 1].numpy()
    nonfraud_emb = train_out["transaction"][train_data["transaction"].y == 0].numpy()

# t-SNE visualization
combined = np.vstack([fraud_emb[:1000], nonfraud_emb[:1000]])
labels = np.array([1]*1000 + [0]*1000)
tsne = TSNE(n_components=2, random_state=42)
embedded = tsne.fit_transform(combined)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.scatter(embedded[labels==0, 0], embedded[labels==0, 1], c='blue', alpha=0.5, label='Non-Fraud')
plt.scatter(embedded[labels==1, 0], embedded[labels==1, 1], c='red', alpha=0.5, label='Fraud')
plt.title("GNN Embeddings: Fraud vs Non-Fraud")
plt.legend()
plt.show()

# Save model

In [ ]:
import pickle
import torch
from google.colab import files

# 1. Save PyTorch GNN weights
torch.save(encoder.state_dict(), 'gnn_encoder_weights.pth')
torch.save(head.state_dict(), 'gnn_head_weights.pth')

# 2. Save XGBoost model
model.save_model('xgboost_model.json')

# 3. Save the Scalers, ID Mappings, AND historical node-feature tensors
#    (the two new keys fix the KeyError on load)
preprocessing_artifacts = {
    'scaler': scaler,
    'cust_scaler': cust_scaler,
    'merch_scaler': merch_scaler,
    'emb_scaler': emb_scaler,
    'customer_id_map': customer_id_map,
    'user_id_map': user_id_map,
    'user_scaler': user_scaler,
    'train_user_x': train_user_x.detach().cpu(),
    'merchant_id_map': merchant_id_map,
    'feature_cols': feature_cols,
    'continuous_cols': continuous_cols,
    'xgb_feature_cols': X_train_numeric.columns.tolist(),
    'best_threshold': best_thresh,
    # Updated to match the new instantiated architecture
    'hidden_channels': 128,
    'out_channels': 64,
    'node_types': list(train_data.node_types),
    'edge_types': list(train_data.edge_types),
    'mcc_freq': mcc_freq.to_dict(),
    # Removed 'fraud_rate' and 'avg_hour' as they are no longer in the feature set
    'node_feature_cols': [
        "txn_count", "mean_amount", "std_amount", "night_txn_ratio",
        "online_ratio", "swipe_ratio", "mcc_diversity", "error_rate"
    ],
    'train_customer_x': train_customer_x.detach().cpu(),
    'train_merchant_x': train_merchant_x.detach().cpu(),
}

with open('preprocessing_artifacts.pkl', 'wb') as f:
    pickle.dump(preprocessing_artifacts, f)

# Also persist the tensors as standalone .pt files (robust fallback if
# the pickle ever fails to unpickle across torch versions)
torch.save(train_customer_x.detach().cpu(), 'train_customer_x.pt')
torch.save(train_merchant_x.detach().cpu(), 'train_merchant_x.pt')

# Save the historical node feature tensors!

print("All models and artifacts saved!")

files.download('gnn_encoder_weights.pth')
files.download('gnn_head_weights.pth')
files.download('xgboost_model.json')
files.download('preprocessing_artifacts.pkl')
files.download('train_customer_x.pt')
files.download('train_merchant_x.pt')